# Day 17 Assignment: House Price Prediction using Artificial Neural Networks (ANNs) 🏠📉

Welcome to the assignment notebook for Day 17! In this notebook, we apply our knowledge of Deep Learning and Artificial Neural Networks (ANNs) to a regression task: predicting house prices using the cleaned King County House Price dataset.

We will build a Regression ANN using TensorFlow/Keras to estimate house prices based on key features.

In [ ]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
# Step 2: Load the dataset
# We are loading the cleaned King County House Price dataset
df = pd.read_csv('../../../kaggle-Dataset-Uploaded/02_House_Price_Prediction/kc_house_clean_data.csv')
df.head()

In [ ]:
# Step 3: Splitting features and target label
# Features: bedrooms, bathrooms, sqft_living, sqft_lot, floors
# Target: price
X = df[['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors']]
y = df['price']

# Split data into Train and Test sets (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features using StandardScaler
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

# Scale target using StandardScaler to prevent exploding gradients and ensure numerical stability
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

print("Training Features (Scaled, first 5 rows):\n", X_train[:5])
print("\nTraining Target (Scaled, first 5 rows):\n", y_train_scaled[:5])

In [ ]:
# Step 4: Build the Deep Learning Model (ANN Regression)
# For regression, the output layer has 1 neuron and no activation function (linear output)
model = Sequential([
    Dense(32, activation="relu", input_shape=(5,)), # First Hidden Layer with 32 Neurons
    Dense(16, activation="relu"),                  # Second Hidden Layer with 16 Neurons
    Dense(1)                                       # Output Layer for Regression
])

# Display the model summary
model.summary()

In [ ]:
# Step 5: Compile the model
# For regression, we use Mean Squared Error (MSE) loss
# We will track Mean Absolute Error (MAE) as a metric
model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mae"]
)

In [ ]:
# Step 6: Train the model
# Since the dataset has ~18k rows (14.4k training), we train with a batch size of 32 for 50 epochs
history = model.fit(
    X_train,
    y_train_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Step 7: Evaluate the model
# Evaluate using the scaled test data
test_loss, test_mae = model.evaluate(X_test, y_test_scaled, verbose=1)

# Convert scaled MAE back to USD for readability
original_mae = test_mae * scaler_y.scale_[0]

print("\nTest Loss (Scaled MSE):", test_loss)
print("Test MAE (Scaled):", test_mae)
print(f"Test MAE in original scale: ${original_mae:,.2f} USD")

In [ ]:
# Step 8: Predict for a new house
# Imagine a new house with: 3 bedrooms, 2 bathrooms, 1800 sqft living space, 5000 sqft lot size, and 2 floors
new_house = [[3, 2, 1800, 5000, 2.0]]

# Scale the input features using the same feature scaler
new_house_scaled = scaler_X.transform(new_house)

# Predict the scaled price
predicted_price_scaled = model.predict(new_house_scaled)

# Inverse transform the prediction back to original scale (USD)
predicted_price = scaler_y.inverse_transform(predicted_price_scaled)

print(f"\nPredicted House Price: ${predicted_price[0][0]:,.2f} USD")